# 04 — Train: GRU/LSTM direct forecaster

Fits one fixed recurrent candidate for the configured target station and evaluates it once on the sealed test feature artifact. The default is a GRU; change `CELL_TYPE` to `"lstm"` to run the same MVP with an LSTM.

**Inputs:** train-derived and test-derived Stage-3 feature artifacts  
**Outputs:** in-notebook loss curve, prediction preview, and native-unit test metrics only

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

The configuration intentionally defines one candidate per execution. It has no validation split, early stopping, tuning, model persistence, or experiment tracking.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from src.config import FORECAST_HORIZON_HOURS, TARGET_STATION_ID
from src.feature_engineering import feature_column_names, target_column_names

PROCESSED_DIR = Path("data/processed")
CELL_TYPE = "gru"
SEQUENCE_LENGTH = 168
HIDDEN_UNITS = 64
EPOCHS = 50
BATCH_SIZE = 64
LEARNING_RATE = 0.001
SEED = 42
PREDICTION_PREVIEW_ORIGINS = 5
FEATURE_COLUMNS = list(feature_column_names())
TARGET_COLUMNS = list(target_column_names())

if CELL_TYPE not in {"gru", "lstm"}:
    raise ValueError(f"CELL_TYPE must be 'gru' or 'lstm'; got {CELL_TYPE!r}")
if len(TARGET_COLUMNS) != FORECAST_HORIZON_HOURS:
    raise ValueError("Configured target columns do not match FORECAST_HORIZON_HOURS")

DEVICE = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

def seed_everything(seed: int) -> None:
    """Seed the fixed MVP training run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
display(
    pd.DataFrame(
        [
            {
                "cell_type": CELL_TYPE,
                "sequence_length_hours": SEQUENCE_LENGTH,
                "hidden_units": HIDDEN_UNITS,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "seed": SEED,
                "device": str(DEVICE),
            }
        ]
    )
)

## Feature-contract and sequence helpers

Each physical split is checked independently. A sample contains 168 complete, finite predictor rows ending at its issue time; targets are used only when Stage 3 marked that issue time `target_valid`. Consequently, the test split never borrows history from the training split.

In [ ]:
def read_feature_artifact(
    path: Path, *, station_id: str, artifact_name: str
) -> pd.DataFrame:
    """Load one Stage-3 feature artifact and validate its model contract."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing feature artifact for {station_id}: {path}")

    frame = pd.read_parquet(path).copy()
    required = {
        "timestamp", "station_id", "target_valid", *FEATURE_COLUMNS, *TARGET_COLUMNS
    }
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )
    if frame.empty:
        raise ValueError(f"{station_id} {artifact_name} artifact is empty")
    if frame["station_id"].dropna().unique().tolist() != [station_id]:
        raise ValueError(
            f"{artifact_name} artifact must contain only station {station_id!r}"
        )

    timestamps = pd.DatetimeIndex(frame["timestamp"])
    if timestamps.tz is None:
        raise ValueError(f"{station_id} {artifact_name} timestamps must be timezone-aware")
    if timestamps.has_duplicates or not timestamps.is_monotonic_increasing:
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be unique and increasing"
        )
    expected_grid = pd.date_range(timestamps[0], periods=len(timestamps), freq="h")
    if not timestamps.equals(expected_grid):
        raise ValueError(
            f"{station_id} {artifact_name} timestamps must be a contiguous hourly grid"
        )
    if frame["target_valid"].isna().any():
        raise ValueError(f"{station_id} {artifact_name} target_valid contains nulls")

    numeric_columns = [*FEATURE_COLUMNS, *TARGET_COLUMNS]
    try:
        numeric_values = frame[numeric_columns].to_numpy(dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(
            f"{station_id} {artifact_name} artifact has non-numeric required values"
        ) from error
    if np.isinf(numeric_values).any():
        raise ValueError(
            f"{station_id} {artifact_name} artifact has non-finite required values"
        )
    valid_targets = frame["target_valid"].eq(True)
    if frame.loc[valid_targets, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    return frame


def build_sequences(
    frame: pd.DataFrame, *, sequence_length: int, station_id: str, artifact_name: str
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Build split-local recurrent samples from complete predictor windows."""
    if sequence_length < 1:
        raise ValueError("sequence_length must be at least 1")

    predictors = frame[FEATURE_COLUMNS].to_numpy(dtype=float)
    targets = frame[TARGET_COLUMNS].to_numpy(dtype=float)
    finite_predictor_rows = np.isfinite(predictors).all(axis=1)
    complete_windows = (
        pd.Series(finite_predictor_rows, index=frame.index)
        .rolling(sequence_length, min_periods=sequence_length)
        .sum()
        .eq(sequence_length)
        .to_numpy()
    )
    eligible = (
        complete_windows
        & frame["target_valid"].eq(True).to_numpy()
        & np.isfinite(targets).all(axis=1)
    )
    origin_positions = np.flatnonzero(eligible)
    if not len(origin_positions):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has no eligible {sequence_length}-hour sequences"
        )

    sequences = np.stack(
        [predictors[position - sequence_length + 1 : position + 1] for position in origin_positions]
    )
    return sequences, targets[origin_positions], frame.iloc[origin_positions].copy()


def metric_tables(
    actual: np.ndarray, predictions: np.ndarray, *, station_id: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Calculate aggregate and horizon-specific MAE/RMSE in water-level units."""
    aggregate = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "scored_issue_times": len(actual),
                "scored_values": actual.size,
                "mae": mean_absolute_error(actual.ravel(), predictions.ravel()),
                "rmse": root_mean_squared_error(actual.ravel(), predictions.ravel()),
            }
        ]
    )
    per_horizon = pd.DataFrame(
        [
            {
                "station_id": station_id,
                "horizon_hours": horizon,
                "target": target,
                "mae": mean_absolute_error(actual[:, horizon - 1], predictions[:, horizon - 1]),
                "rmse": root_mean_squared_error(
                    actual[:, horizon - 1], predictions[:, horizon - 1]
                ),
            }
            for horizon, target in enumerate(TARGET_COLUMNS, start=1)
        ]
    )
    return aggregate, per_horizon


def prediction_preview(
    origins: pd.DataFrame, actual: np.ndarray, predictions: np.ndarray
) -> pd.DataFrame:
    """Return a long-form preview for the first configured number of forecast origins."""
    preview_count = min(PREDICTION_PREVIEW_ORIGINS, len(origins))
    horizon_hours = np.arange(1, FORECAST_HORIZON_HOURS + 1)
    issue_timestamps = origins["timestamp"].iloc[:preview_count].to_numpy()
    return pd.DataFrame(
        {
            "issue_timestamp": np.repeat(issue_timestamps, FORECAST_HORIZON_HOURS),
            "target_timestamp": np.repeat(issue_timestamps, FORECAST_HORIZON_HOURS)
            + pd.to_timedelta(np.tile(horizon_hours, preview_count), unit="h"),
            "horizon_hours": np.tile(horizon_hours, preview_count),
            "actual_water_level": actual[:preview_count].ravel(),
            "predicted_water_level": predictions[:preview_count].ravel(),
        }
    )

## Load split-local samples

The test sample builder receives only the sealed test artifact. Its first 167 rows therefore cannot become origins, even though preceding training rows exist on the overall timeline.

In [ ]:
station_id = TARGET_STATION_ID
train_path = PROCESSED_DIR / f"{station_id}_train_features.parquet"
test_path = PROCESSED_DIR / f"{station_id}_test_features.parquet"
train_features = read_feature_artifact(
    train_path, station_id=station_id, artifact_name="train"
)
test_features = read_feature_artifact(
    test_path, station_id=station_id, artifact_name="test"
)

train_sequences, train_targets, train_origins = build_sequences(
    train_features,
    sequence_length=SEQUENCE_LENGTH,
    station_id=station_id,
    artifact_name="train",
)
test_sequences, test_targets, test_origins = build_sequences(
    test_features,
    sequence_length=SEQUENCE_LENGTH,
    station_id=station_id,
    artifact_name="test",
)

assert test_origins.index.min() >= test_features.index[SEQUENCE_LENGTH - 1]
display(
    pd.DataFrame(
        [
            {
                "station_id": station_id,
                "train_sequences": len(train_sequences),
                "test_sequences": len(test_sequences),
                "sequence_shape": tuple(train_sequences.shape[1:]),
                "excluded_test_warmup_rows": SEQUENCE_LENGTH - 1,
            }
        ]
    )
)

## Standardize, fit, and predict

The predictor scaler is fitted on every predictor value contained in training windows; the target scaler is fitted on the training target vectors. Test values use those fitted scalers, and predictions are inverse-transformed before evaluation.

In [ ]:
predictor_scaler = StandardScaler()
target_scaler = StandardScaler()

train_sequence_shape = train_sequences.shape
test_sequence_shape = test_sequences.shape
train_predictors = predictor_scaler.fit_transform(
    train_sequences.reshape(-1, len(FEATURE_COLUMNS))
).reshape(train_sequence_shape)
test_predictors = predictor_scaler.transform(
    test_sequences.reshape(-1, len(FEATURE_COLUMNS))
).reshape(test_sequence_shape)
train_targets_scaled = target_scaler.fit_transform(train_targets)

class RecurrentForecaster(nn.Module):
    """One-layer direct multi-horizon GRU or LSTM forecaster."""

    def __init__(self, *, cell_type: str, input_size: int, hidden_units: int) -> None:
        super().__init__()
        recurrent_class = nn.GRU if cell_type == "gru" else nn.LSTM
        self.recurrent = recurrent_class(
            input_size=input_size,
            hidden_size=hidden_units,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_units, FORECAST_HORIZON_HOURS)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        outputs, _ = self.recurrent(inputs)
        return self.head(outputs[:, -1, :])


model = RecurrentForecaster(
    cell_type=CELL_TYPE,
    input_size=len(FEATURE_COLUMNS),
    hidden_units=HIDDEN_UNITS,
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_function = nn.MSELoss()

train_dataset = TensorDataset(
    torch.tensor(train_predictors, dtype=torch.float32),
    torch.tensor(train_targets_scaled, dtype=torch.float32),
)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=loader_generator
)

epoch_losses: list[float] = []
for _ in range(EPOCHS):
    model.train()
    weighted_loss = 0.0
    sample_count = 0
    for batch_predictors, batch_targets in train_loader:
        batch_predictors = batch_predictors.to(DEVICE)
        batch_targets = batch_targets.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_function(model(batch_predictors), batch_targets)
        loss.backward()
        optimizer.step()
        weighted_loss += loss.item() * len(batch_predictors)
        sample_count += len(batch_predictors)
    epoch_losses.append(weighted_loss / sample_count)

model.eval()
with torch.no_grad():
    scaled_test_predictions = (
        model(torch.tensor(test_predictors, dtype=torch.float32, device=DEVICE))
        .cpu()
        .numpy()
    )
test_predictions = target_scaler.inverse_transform(scaled_test_predictions)
if test_predictions.shape != (len(test_origins), FORECAST_HORIZON_HOURS):
    raise RuntimeError(
        f"Expected ({len(test_origins)}, {FORECAST_HORIZON_HOURS}) predictions; "
        f"got {test_predictions.shape}"
    )

## Training diagnostic and test evaluation

The loss curve is an inline training diagnostic. All reported error metrics and previews are in the original water-level units.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), epoch_losses)
plt.xlabel("Epoch")
plt.ylabel("Training MSE (scaled targets)")
plt.title(f"{CELL_TYPE.upper()} training loss")
plt.grid(alpha=0.3)
plt.show()

aggregate_metrics, per_horizon_metrics = metric_tables(
    test_targets, test_predictions, station_id=station_id
)
print(f"{CELL_TYPE.upper()} test results for {station_id}")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_origins, test_targets, test_predictions))